In [8]:
from backtesting.test import GOOG

import math

In [2]:
GOOG.head()

,Open,High,Low,Close,Volume
2004-08-19,100.00,104.06,95.96,100.34,22351900
2004-08-20,101.01,109.08,100.50,108.31,11428600
2004-08-23,110.75,113.48,109.05,109.40,9137200
2004-08-24,111.24,111.60,103.57,104.87,7631300
2004-08-25,104.96,108.00,103.88,106.00,4598900


In [3]:
GOOG.tail()

,Open,High,Low,Close,Volume
2013-02-25,802.3,808.41,790.49,790.77,2303900
2013-02-26,795.0,795.95,784.40,790.13,2202500
2013-02-27,794.8,804.75,791.11,799.78,2026100
2013-02-28,801.1,806.99,801.03,801.20,2265800
2013-03-01,797.8,807.14,796.15,806.19,2175400


In [13]:
def Momentum(values, n):
    """
    Bet in the direction of the recent price change
    """
    result = []
    for i, v in enumerate(values):
        if i <= n - 1:
            result.append(float('nan'))
            continue
        mom_direction = math.copysign(1, values[i] - values[i-n])
        result.append(mom_direction)
    return result

In [63]:
from backtesting import Strategy
from backtesting.lib import crossover

class MomMr(Strategy):
    # Lookback
    bars = 1
    is_mom = True
    
    def init(self):
        self.mom = self.I(Momentum, self.data.Close, self.bars)
    
    def next(self):
        # Note to self: The last array value (e.g. mom[-1]) is always the most recent value.
        should_go_long = (self.mom[-1] if self.is_mom else -self.mom[-1]) > 0
        if math.isnan(should_go_long):
            return
        # NOTE: tricky part with `not self.position.is_short`
        # print(self.position, " ", self.position.is_long, " ", self.position.is_short, " ", should_go_long)
        if (not self.position or self.position.is_long) and not should_go_long:
            self.position.close()
            self.sell()
        elif (not self.position or self.position.is_short) and should_go_long:
            self.position.close()
            self.buy()

In [88]:
from backtesting import Backtest

# NOTE: increase cash!
bt = Backtest(GOOG, MomMr, cash=100_000, margin=1, commission=.001, finalize_trades=True) # .002
stats = bt.run()
stats

Start                     2004-08-19 00:00:00
End                       2013-03-01 00:00:00
Duration                   3116 days 00:00:00
Exposure Time [%]                    99.90689
Equity Final [$]                  11241.58736
Equity Peak [$]                  107982.74045
Commissions [$]                   78684.73264
Return [%]                          -88.75841
Buy & Hold Return [%]               703.45824
Return (Ann.) [%]                   -22.62642
Volatility (Ann.) [%]                26.57144
CAGR [%]                            -22.60012
Sharpe Ratio                         -0.85153
Sortino Ratio                        -0.96194
Calmar Ratio                         -0.25106
Alpha [%]                          -105.73655
Beta                                  0.02414
Max. Drawdown [%]                     -90.122
Avg. Drawdown [%]                   -49.87697
Max. Drawdown Duration     3084 days 00:00:00
Avg. Drawdown Duration     1555 days 00:00:00
# Trades                          

In [89]:
# NOTE: interesting! If it's so bad, can we just do the opposite?
a = bt.plot(superimpose=False)

/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_scroll' property; using the latest value
  fig = gridplot(


In [90]:
%%time

stats = bt.optimize(bars=[1,2,3,5,7,10,14], # NOTE: 7 was best so let's extend!! TODO more
                    is_mom=[True, False],
                    maximize='Equity Final [$]',)
                    #constraint=lambda param: param.n1 < param.n2)
stats

/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/backtesting.py:1639: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


CPU times: user 357 ms, sys: 42.9 ms, total: 399 ms
Wall time: 378 ms


Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/psm_a21ca4bb'
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/psm_c5f0f283'
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/psm_a1b41f28'
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyErr

Start                     2004-08-19 00:00:00
End                       2013-03-01 00:00:00
Duration                   3116 days 00:00:00
Exposure Time [%]                    99.30168
Equity Final [$]                 672427.70273
Equity Peak [$]                  680729.39813
Commissions [$]                  145492.54727
Return [%]                           572.4277
Buy & Hold Return [%]               688.06452
Return (Ann.) [%]                    25.06717
Volatility (Ann.) [%]                41.53686
CAGR [%]                             25.03011
Sharpe Ratio                          0.60349
Sortino Ratio                         1.19702
Calmar Ratio                          0.59318
Alpha [%]                           513.09571
Beta                                  0.08623
Max. Drawdown [%]                   -42.25898
Avg. Drawdown [%]                    -6.08469
Max. Drawdown Duration      997 days 00:00:00
Avg. Drawdown Duration       55 days 00:00:00
# Trades                          

In [91]:
stats._strategy

<Strategy MomMr(bars=14,is_mom=)>

In [92]:
bt.plot(plot_volume=False, plot_pl=False, superimpose=False)

/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_scroll' property; using the latest value
  fig = gridplot(


GridPlot(id='p5557', ...)